# Carichi e combinazioni — esempio completo

Calcolo delle azioni e delle combinazioni SLU/SLE per un edificio residenziale secondo NTC18.

**Dati geometrici**:
- Edificio residenziale (categoria A)
- Zona vento: 3, altitudine: 250 m s.l.m.
- Zona neve: II, altitudine: 250 m s.l.m.
- Periodo di ritorno di riferimento: 50 anni

In [9]:
import math

from pyntc.actions.loads import unit_weight, variable_load, partition_equivalent_load
from pyntc.actions.wind import (
    wind_base_velocity,
    wind_reference_velocity,
    wind_kinetic_pressure,
    wind_exposure_coefficient,
    wind_pressure,
)
from pyntc.actions.snow import snow_ground_load, snow_shape_coefficient, snow_roof_load
from pyntc.actions.combinations import (
    combination_coefficients,
    partial_safety_factors,
    slu_combination,
    sle_characteristic_combination,
    sle_quasi_permanent_combination,
)

## 1. Carichi permanenti — NTC18 §3.1

In [ ]:
# Valori attesi — NTC18 Tab. 3.1.I e §3.1.3
expected_gamma_ca = 25.0   # Tab. 3.1.I — calcestruzzo armato [kN/m³]
expected_G1       = 25.0 * 0.20   # = 5.0 kN/m²
expected_g2_part  = 0.80   # §3.1.3 — peso_per_m 1.5 kN/m → scaglione (1.0, 2.0] kN/m

gamma_ca = unit_weight("calcestruzzo_armato")
spessore_solaio = 0.20  # m
G1 = gamma_ca * spessore_solaio
print(f"Peso proprio solaio G1 = {G1:.2f} kN/m²")

G2_pavimento  = 1.0   # kN/m²  pavimento + massetto
G2_partizioni = partition_equivalent_load(weight_per_meter=1.5)
G2 = G2_pavimento + G2_partizioni
print(f"Partizioni equivalenti = {G2_partizioni:.2f} kN/m²")
print(f"Carichi permanenti portati G2 = {G2:.2f} kN/m²")

assert math.isclose(gamma_ca,    expected_gamma_ca, rel_tol=1e-3)
assert math.isclose(G1,          expected_G1,       rel_tol=1e-3)
assert math.isclose(G2_partizioni, expected_g2_part, rel_tol=1e-3)

Peso proprio solaio G1 = 5.00 kN/m²


TypeError: partition_equivalent_load() got an unexpected keyword argument 'weight_per_m'. Did you mean 'weight_per_meter'?

## 2. Carichi variabili — NTC18 §3.1, Tab. 3.1.II

In [ ]:
# Valori attesi — NTC18 Tab. 3.1.II categoria A (residenziale)
expected_qk, expected_Qk, expected_Hk = 2.0, 2.0, 1.0

qk, Qk, Hk = variable_load("A")
print(f"Carico distribuito qk = {qk:.1f} kN/m²")
print(f"Carico concentrato Qk = {Qk:.1f} kN")
print(f"Carico orizzontale Hk = {Hk:.1f} kN/m")

assert (qk, Qk, Hk) == (expected_qk, expected_Qk, expected_Hk)

## 3. Azione del vento — NTC18 §3.3

In [ ]:
zona = 3
altitudine = 250.0  # m s.l.m.
z = 10.0            # altezza dal suolo [m]
cat_esposizione = 3 # categoria di esposizione

# Valori attesi calcolati dalle formule NTC18
# Tab.3.3.I zona 3: v_b0=27.0, a_0=500m → alt=250 < a_0 → c_a=1.0 → v_b=27.0 m/s
expected_v_b = 27.0
# [3.3.2]: v_r = v_b * c_R; T_R=50 anni → c_R ≈ 1.0 (periodo di riferimento)
# Tab.3.3.II cat 3: k_r=0.20, z_0=0.10, z_min=5 → z=10 > z_min
_kr, _z0 = 0.20, 0.10
_lnz = math.log(z / _z0)
expected_c_e = _kr**2 * _lnz * (7 + _lnz)
# [3.3.6]: q_b = 0.5 * rho * v_r² / 1000

v_b = wind_base_velocity(zona, altitudine)
v_r = wind_reference_velocity(zona, altitudine, return_period=50)
q_b = wind_kinetic_pressure(v_r)
c_e = wind_exposure_coefficient(z, cat_esposizione)
expected_q_b = 0.5 * 1.25 * v_r**2 / 1000  # usa v_r già verificato

p_sopravento = wind_pressure(q_b, c_e, c_p=+0.8)
p_sottovento = wind_pressure(q_b, c_e, c_p=-0.5)
Qw = p_sopravento - p_sottovento
expected_p_sov = q_b * c_e * 0.8
expected_Qw    = q_b * c_e * (0.8 + 0.5)

print(f"Velocità base v_b      = {v_b:.2f} m/s")
print(f"Velocità riferimento v_r = {v_r:.2f} m/s")
print(f"Pressione cinetica q_b  = {q_b:.3f} kN/m²")
print(f"Coeff. esposizione c_e  = {c_e:.3f}")
print(f"Pressione netta vento   = {Qw:.3f} kN/m²")

assert math.isclose(v_b, expected_v_b,  rel_tol=1e-3)
assert math.isclose(q_b, expected_q_b,  rel_tol=1e-3)
assert math.isclose(c_e, expected_c_e,  rel_tol=1e-3)
assert math.isclose(p_sopravento, expected_p_sov, rel_tol=1e-3)
assert math.isclose(Qw, expected_Qw,    rel_tol=1e-3)

## 4. Azione della neve — NTC18 §3.4

In [ ]:
zona_neve = "II"
# Valori attesi — NTC18 Tab.3.4.I zona II, §3.4.2
# alt=250 > 200m → q_sk = 0.85 * [1 + (a_s/481)²]
expected_q_sk = 0.85 * (1 + (altitudine / 481)**2)
# Tab.3.4.II: mu_1 = 0.8 per alpha = 0°
expected_mu  = 0.8
expected_qs  = expected_q_sk * expected_mu * 1.0 * 1.0   # C_e=C_t=1

q_sk = snow_ground_load(zona_neve, altitudine)
mu   = snow_shape_coefficient(alpha=0.0)
C_e  = 1.0
C_t  = 1.0
qs   = snow_roof_load(q_sk, mu, C_e, C_t)

print(f"Carico neve al suolo q_sk = {q_sk:.3f} kN/m²")
print(f"Coeff. forma μ            = {mu:.2f}")
print(f"Carico neve in copertura  = {qs:.3f} kN/m²")

assert math.isclose(q_sk, expected_q_sk, rel_tol=1e-3)
assert math.isclose(mu,   expected_mu,   rel_tol=1e-3)
assert math.isclose(qs,   expected_qs,   rel_tol=1e-3)

## 5. Coefficienti di combinazione — NTC18 §2.5.3, Tab. 2.5.I

In [ ]:
# Valori attesi — NTC18 Tab. 2.5.I
expected_psi_A    = (0.7, 0.5, 0.3)   # cat. A residenziale
expected_psi_wind = (0.6, 0.2, 0.0)   # vento
expected_psi_snow = (0.5, 0.2, 0.0)   # neve ≤ 1000 m

psi0_A, psi1_A, psi2_A       = combination_coefficients("A")
psi0_w, psi1_w, psi2_w       = combination_coefficients("wind")
psi0_s, psi1_s, psi2_s       = combination_coefficients("snow_low")

print(f"ψ residenziale  ψ0={psi0_A}, ψ1={psi1_A}, ψ2={psi2_A}")
print(f"ψ vento         ψ0={psi0_w}, ψ1={psi1_w}, ψ2={psi2_w}")
print(f"ψ neve          ψ0={psi0_s}, ψ1={psi1_s}, ψ2={psi2_s}")

assert (psi0_A, psi1_A, psi2_A) == expected_psi_A
assert (psi0_w, psi1_w, psi2_w) == expected_psi_wind
assert (psi0_s, psi1_s, psi2_s) == expected_psi_snow

## 6. Combinazioni SLU — NTC18 §2.5.3, Formula [2.5.1]

In [ ]:
# Valori attesi — [2.5.1]: γ_G1=1.3, γ_G2=1.5, γ_Q=1.5 (approccio A1, sfavorevole)
# La funzione prova ogni Q come dominante → max delle 3 ipotesi
_gG1, _gG2, _gQ = 1.3, 1.5, 1.5
expected_E_slu = max(
    _gG1*G1 + _gG2*G2 + _gQ*qk  + _gQ*(psi0_w*Qw + psi0_s*qs),   # qk dominante
    _gG1*G1 + _gG2*G2 + _gQ*Qw  + _gQ*(psi0_A*qk  + psi0_s*qs),   # Qw dominante
    _gG1*G1 + _gG2*G2 + _gQ*qs  + _gQ*(psi0_A*qk  + psi0_w*Qw),   # qs dominante
)

# Coefficienti γ — approccio A1
gamma_G1 = partial_safety_factors("G1", favorable=False, approach="A1")
gamma_G2 = partial_safety_factors("G2", favorable=False, approach="A1")
gamma_Q  = partial_safety_factors("Q",  favorable=False, approach="A1")

# SLU — funzione sceglie automaticamente la combinazione più gravosa
E_slu = slu_combination(
    G1=G1, G2=G2,
    Q=[qk, Qw, qs],
    categories=["A", "wind", "snow_low"],
    approach="A1",
)
print(f"γ_G1={gamma_G1}, γ_G2={gamma_G2}, γ_Q={gamma_Q}")
print(f"SLU di progetto = {E_slu:.3f} kN/m²")

assert math.isclose(gamma_G1, _gG1, rel_tol=1e-3)
assert math.isclose(gamma_G2, _gG2, rel_tol=1e-3)
assert math.isclose(gamma_Q,  _gQ,  rel_tol=1e-3)
assert math.isclose(E_slu, expected_E_slu, rel_tol=1e-3)

## 7. Combinazioni SLE — NTC18 §2.5.3, Formule [2.5.3]–[2.5.5]

In [ ]:
# Valori attesi — [2.5.2] SLE caratteristica: G1+G2+Q_dom+Σ(ψ0i*Qi)
expected_E_sle_car = max(
    G1 + G2 + qk + psi0_w*Qw + psi0_s*qs,
    G1 + G2 + Qw + psi0_A*qk + psi0_s*qs,
    G1 + G2 + qs + psi0_A*qk + psi0_w*Qw,
)
# [2.5.4] SLE quasi-permanente: G1+G2+Σ(ψ2i*Qi) — psi2_w=psi2_s=0
expected_E_sle_qp = G1 + G2 + psi2_A*qk + psi2_w*Qw + psi2_s*qs

E_sle_car = sle_characteristic_combination(
    G1=G1, G2=G2,
    Q=[qk, Qw, qs],
    categories=["A", "wind", "snow_low"],
)
E_sle_qp = sle_quasi_permanent_combination(
    G1=G1, G2=G2,
    Q=[qk, Qw, qs],
    categories=["A", "wind", "snow_low"],
)

print(f"SLE caratteristica    = {E_sle_car:.3f} kN/m²")
print(f"SLE quasi-permanente  = {E_sle_qp:.3f} kN/m²")

assert math.isclose(E_sle_car, expected_E_sle_car, rel_tol=1e-3)
assert math.isclose(E_sle_qp,  expected_E_sle_qp,  rel_tol=1e-3)

## Riepilogo

In [ ]:
print("=" * 45)
print("RIEPILOGO AZIONI E COMBINAZIONI")
print("=" * 45)
print(f"G1  (peso proprio)       = {G1:.2f}  kN/m²")
print(f"G2  (permanenti portati) = {G2:.2f}  kN/m²")
print(f"qk  (residenziale)       = {qk:.2f}  kN/m²")
print(f"Qw  (vento netto)        = {Qw:.3f} kN/m²")
print(f"qs  (neve)               = {qs:.3f} kN/m²")
print("-" * 45)
print(f"SLU di progetto          = {E_slu:.3f} kN/m²")
print(f"SLE caratteristica       = {E_sle_car:.3f} kN/m²")
print(f"SLE quasi-permanente     = {E_sle_qp:.3f} kN/m²")
print("=" * 45)